In [1]:
!pip install datasets

In [2]:
dataset= ["Book5.txt"]

In [3]:
import os
import json

raw_text = "" # Initialize raw_text as an empty string

for filename in dataset: # Iterate directly over the list of filenames
  with open(filename, 'r', encoding='utf-8') as f:
    raw_text += f.read() # Use += to append the content

print(f"Length of dataset in characters: {len(raw_text)}")

Length of dataset in characters: 1608763


In [4]:
dataset= ["Book5.txt"]

In [5]:
import os
import json

raw_text = "" # Initialize raw_text as an empty string

for filename in dataset: # Iterate directly over the list of filenames
  with open(filename, 'r', encoding='utf-8') as f:
    raw_text += f.read() # Use += to append the content

print(f"Length of dataset in characters: {len(raw_text)}")

Length of dataset in characters: 1608763


In [6]:
print("Total No. of Characters", len(raw_text))
print(raw_text[:500])

Total No. of Characters 1608763
HARRY 

POTTER 




I 




DUDLEY DEMENTED 

The hottest day of the summer so far was drawing to 
a close and a drowsy silence lay over the large, square 
houses of Privet Drive. Cars that were usually 
gleaming stood dusty in their drives and lawns that 
were once emerald green lay parched and yellowing; 
the use of hosepipes had been banned due to 
drought. Deprived of their usual car-washing and 
lawn-mowing pursuits, the inhabitants of Privet Drive 
had retreated into the shade of their cool


In [7]:
!pip install tiktoken

In [8]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")
enc_text_indices = tokenizer.encode(raw_text)
print("Total No. of Tokens", len(enc_text_indices))
print(enc_text_indices[:100])

Total No. of Tokens 405204
[39, 77964, 4815, 47, 1831, 4292, 77425, 40, 77425, 35, 4760, 54949, 423, 16837, 1507, 4815, 791, 38391, 1938, 315, 279, 7474, 779, 3117, 574, 13633, 311, 720, 64, 3345, 323, 264, 294, 1849, 88, 21847, 11203, 927, 279, 3544, 11, 9518, 720, 37841, 315, 15801, 295, 16542, 13, 36231, 430, 1051, 6118, 720, 3491, 6605, 14980, 77973, 304, 872, 20722, 323, 2383, 4511, 430, 720, 52898, 3131, 991, 25380, 6307, 11203, 66302, 291, 323, 14071, 287, 26, 720, 1820, 1005, 315, 38600, 87820, 1047, 1027, 21501, 4245, 311, 720, 67, 6478, 13, 1611, 652, 2270, 315, 872, 13783, 1841]


In [9]:
ex = tokenizer.encode("hello hi ")
print(ex)

string = tokenizer.decode(ex)
print(string)

[15339, 15960, 220]
hello hi 


In [10]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [11]:
GPT_CONFIG_124M = {
    "vocab_size": 100277,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 2,  #12        # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [12]:
def create_dataloader_v1(txt, batch_size = GPT_CONFIG_124M['n_heads'], max_length = GPT_CONFIG_124M['context_length'],
                         stride = GPT_CONFIG_124M['context_length'], shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("cl100k_base")
    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    # Check if the tokenized text is long enough to create sequences
    if len(token_ids) < max_length:
        print(f"Warning: The provided text is too short (length: {len(token_ids)}) to create sequences of length {max_length} with a stride of {stride}.")
        print("Please provide a longer text or reduce max_length/stride.")
        return None  # Return None or an empty dataloader

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [13]:
dataloader = create_dataloader_v1(raw_text, batch_size = GPT_CONFIG_124M['n_heads'],
                                  max_length = GPT_CONFIG_124M['context_length'],
                                  stride=GPT_CONFIG_124M['context_length'], shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

print(inputs.shape)

Inputs:
 tensor([[   39, 77964,  4815,  ...,    13, 19292, 18396],
        [ 2001,  1306,   264,  ..., 54499,    13,  1054]])

Targets:
 tensor([[77964,  4815,    47,  ..., 19292, 18396,  2001],
        [ 1306,   264,   720,  ...,    13,  1054,  7131]])
torch.Size([2, 1024])


In [14]:
import torch
vocab_size = GPT_CONFIG_124M['vocab_size']
output_dim = GPT_CONFIG_124M['emb_dim']
torch.manual_seed(111)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print(embedding_layer.weight)
print(embedding_layer.weight.shape)

Parameter containing:
tensor([[-1.2634e+00, -2.3912e-01,  3.1981e-01,  ...,  1.9824e+00,
         -5.6003e-01, -1.1854e+00],
        [-1.0768e+00,  4.6112e-01,  4.5850e-01,  ..., -4.6378e-01,
         -1.2845e-01,  5.0173e-01],
        [ 4.2807e-01,  5.8008e-05,  8.7423e-01,  ..., -1.4174e+00,
         -1.1270e+00,  1.0560e+00],
        ...,
        [-6.0770e-01, -2.5817e-01,  1.1078e+00,  ..., -4.3280e-01,
         -1.9683e-01,  3.9149e-01],
        [ 1.7813e+00, -4.1577e-02,  7.9112e-01,  ...,  5.1642e-01,
         -1.5800e+00,  1.2703e+00],
        [ 1.3022e-02,  1.1854e+00,  1.6086e+00,  ...,  1.6270e-01,
         -7.3789e-01,  7.3316e-01]], requires_grad=True)
torch.Size([100277, 768])


In [15]:
token_embeddings = embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([2, 1024, 768])


In [16]:
max_length = GPT_CONFIG_124M['context_length']
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [17]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([1024, 768])


In [18]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([2, 1024, 768])


In [19]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)

In [20]:
import torch.nn as nn
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [21]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), ## Expansion
            GELU(), ## Activation
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), ## Contraction
        )

    def forward(self, x):
        return self.layers(x)

In [22]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        # 2*4*768
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

In [23]:
import torch
import torch.nn as nn


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [24]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context

    ###Input batch:
 ###tensor([[6109, 3626, 6100,  345],
        ##[6109, 1110, 6622,  257]])

    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond) ### batch, n_tokens, vocab_size

        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Ensure the generated token ID is within the valid vocabulary range
        #idx_next = torch.clamp(idx_next, 0, model.cfg["vocab_size"] - 1)


        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [25]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

In [26]:
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0) #A
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [9906, 11, 358, 1097]
encoded_tensor.shape: torch.Size([1, 4])


In [27]:
model.eval() #A
model = GPTModel(GPT_CONFIG_124M)
out = generate_text_simple(
model=model,
idx=encoded_tensor,
max_new_tokens=6,
context_size=GPT_CONFIG_124M["context_length"]
)
print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[ 9906,    11,   358,  1097, 87229, 82269, 27044, 78589,  7200, 76954]])
Output length: 10


In [28]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I am-su Searches Includesbubble band-Shirt


In [29]:
import tiktoken

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("cl100k_base")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you.'

pill$(' Flux[thread shielding qryatteryacceptableusaha


In [30]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1) # Probability of each token in vocabulary
print(probas.shape) # Shape: (batch_size, num_tokens, vocab_size)

torch.Size([2, 1024, 100277])


In [31]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", token_ids)
print(token_ids.shape)

Token IDs:
 tensor([[[26797],
         [70838],
         [74760],
         ...,
         [94189],
         [76242],
         [21757]],

        [[10037],
         [   40],
         [ 9267],
         ...,
         [44712],
         [56147],
         [30411]]])
torch.Size([2, 1024, 1])


In [32]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Targets batch 1: ARRY 

POTTER 




I 




DUDLEY DEMENTED 

The hottest day of the summer so far was drawing to 
a close and a drowsy silence lay over the large, square 
houses of Privet Drive. Cars that were usually 
gleaming stood dusty in their drives and lawns that 
were once emerald green lay parched and yellowing; 
the use of hosepipes had been banned due to 
drought. Deprived of their usual car-washing and 
lawn-mowing pursuits, the inhabitants of Privet Drive 
had retreated into the shade of their cool houses, 
windows thrown wide in the hope of tempting in a 
nonexistent breeze. The only person left outdoors was 
a teenage boy who was lying flat on his back in a 
flower bed outside number four. 

He was a skinny, black-haired, bespectacled boy who 
had the pinched, slightly unhealthy look of someone 
who has grown a lot in a short space of time. His 
jeans were torn and dirty, his T-shirt baggy and 
faded, and the soles of his trainers were peeling away 
from the uppers. Harr

In [33]:
text_idx = 0
target_probas_1 = probas[text_idx, 0:1023, targets[text_idx, 0:1023]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, 0:1023, targets[text_idx, 0:1023]]
print("Text 2:", target_probas_2)

Text 1: tensor([[1.8353e-05, 1.4301e-05, 4.7141e-06,  ..., 2.3153e-05, 1.6340e-05,
         3.8483e-06],
        [7.0822e-06, 6.9905e-06, 1.2176e-05,  ..., 6.0807e-06, 5.5134e-06,
         9.5206e-06],
        [2.0148e-05, 2.0547e-05, 1.6621e-05,  ..., 8.4921e-06, 8.9802e-06,
         1.1745e-05],
        ...,
        [5.2039e-06, 1.1083e-05, 2.8031e-05,  ..., 3.1076e-05, 9.7161e-06,
         2.5085e-06],
        [2.0648e-05, 8.6492e-06, 9.2083e-06,  ..., 2.5588e-05, 2.2410e-05,
         1.0570e-05],
        [1.8145e-05, 7.6015e-06, 9.5145e-06,  ..., 1.8088e-05, 8.8732e-06,
         1.5385e-05]])
Text 2: tensor([[5.5490e-06, 1.1779e-05, 7.2760e-06,  ..., 2.8345e-06, 1.9016e-05,
         1.2540e-05],
        [1.3860e-05, 1.2974e-05, 1.0229e-05,  ..., 8.1254e-06, 7.3373e-06,
         5.9906e-06],
        [5.5210e-06, 5.2397e-06, 7.9250e-06,  ..., 2.0950e-05, 5.9648e-06,
         6.3329e-06],
        ...,
        [7.5077e-06, 8.8613e-06, 1.0991e-05,  ..., 1.2603e-05, 9.8412e-06,
         

In [34]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

tensor([[-10.9057, -11.1552, -12.2650,  ..., -10.6734, -11.0219, -12.4679],
        [-11.8579, -11.8710, -11.3161,  ..., -12.0104, -12.1083, -11.5621],
        [-10.8124, -10.7928, -11.0048,  ..., -11.6764, -11.6205, -11.3521],
        ...,
        [-11.7996, -11.6338, -11.4184,  ..., -11.2815, -11.5289, -12.4921],
        [-11.5667, -11.7029, -12.0986,  ..., -10.6381, -11.5191, -12.0746],
        [-10.5640, -11.2579, -13.1738,  ..., -12.9155, -10.7061, -11.7122]])


In [35]:
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-11.7065)


In [36]:
# Logits have shape (batch_size, num_tokens, vocab_size)
print("Logits shape:", logits.shape)

# Targets have shape (batch_size, num_tokens)
print("Targets shape:", targets.shape)

Logits shape: torch.Size([2, 1024, 100277])
Targets shape: torch.Size([2, 1024])


In [37]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([2048, 100277])
Flattened targets: torch.Size([2048])


In [38]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([2048, 100277])
Flattened targets: torch.Size([2048])


In [39]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(11.6861)


In [40]:
perplexity = torch.exp(loss)
print(perplexity)

tensor(118902.8281)


In [41]:
# Train/validation ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(raw_text))
train_data = raw_text[:split_idx]
val_data = raw_text[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [42]:
if len(tokenizer.encode(raw_text)) * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "increase the `training_ratio`")

if len(tokenizer.encode(raw_text)) * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")

In [43]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

Train loader:
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 1024]) torch.Size([2, 1024])
torch.Size([2, 102

In [44]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

Training tokens: 364544
Validation tokens: 39936
All tokens: 404480


In [45]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [46]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen, train_accuracies, val_accuracies = [], [], [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        correct_train_total = 0
        tokens_train_total = 0

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            #Accuracy metric:
            logits = model(input_batch.to(device))
            #Compute Loss
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            #Backward and optimize
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel() # Returns the total number of elements (or tokens) in the input_batch.
            global_step += 1
            #Compute training accuracy
            predictions = logits.argmax(dim=-1)
            correct_train = (predictions == target_batch.to(device)).sum().item()
            total_train = target_batch.numel()
            correct_train_total += correct_train
            tokens_train_total += total_train


            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss, train_accuracy, val_accuracy = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                train_accuracies.append(train_accuracy)
                val_accuracies.append(val_accuracy)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train accuracy {train_accuracy:.3f}, Val accuracy {val_accuracy:.3f}")

        train_accuracy = correct_train_total / tokens_train_total
        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, train_accuracies, val_accuracies, track_tokens_seen

In [47]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

In [48]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    correct_train = 0
    total_train = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        # Training loss and accuracy
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        for input_batch, target_batch in train_loader:
            logits = model(input_batch.to(device))
            preds = logits.argmax(dim=-1)
            correct_train += (preds == target_batch.to(device)).sum().item()
            total_train += target_batch.numel()
            if total_train >= eval_iter * input_batch.numel():
                break

        train_accuracy = correct_train / total_train

        # Validation loss and accuracy
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
        for input_batch, target_batch in val_loader:
            logits = model(input_batch.to(device))
            preds = logits.argmax(dim=-1)
            correct_val += (preds == target_batch.to(device)).sum().item()
            total_val += target_batch.numel()
            if total_val >= eval_iter * input_batch.numel():
                break

        val_accuracy = correct_val / total_val

    model.train()
    return train_loss, val_loss, train_accuracy, val_accuracy

In [49]:
device = torch.device("cuda" if torch.cuda.is_available() else "gpu")
model = model.to(device)


# Note:
# Uncommenting the following lines will allow the code to run on Apple Silicon chips, if applicable,
# which is approximately 2x faster than on an Apple CPU (as measured on an M3 MacBook Air).
# However, the resulting loss values may be slightly different.

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using {device} device.")


model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes

Using cuda device.


GPTModel(
  (tok_emb): Embedding(100277, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_featur

In [50]:
import time
start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 1
train_losses, val_losses, train_accuracies, val_accuracies, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# Note:
# Uncomment the following code to show the execution time
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 10.529, Val loss 10.549
Ep 1 (Step 000000): Train accuracy 0.056, Val accuracy 0.057
Ep 1 (Step 000005): Train loss 8.735, Val loss 8.824
Ep 1 (Step 000005): Train accuracy 0.057, Val accuracy 0.057
Ep 1 (Step 000010): Train loss 7.478, Val loss 7.595
Ep 1 (Step 000010): Train accuracy 0.079, Val accuracy 0.081
Ep 1 (Step 000015): Train loss 6.896, Val loss 7.109
Ep 1 (Step 000015): Train accuracy 0.102, Val accuracy 0.101
Ep 1 (Step 000020): Train loss 6.779, Val loss 6.902
Ep 1 (Step 000020): Train accuracy 0.113, Val accuracy 0.113
Ep 1 (Step 000025): Train loss 6.524, Val loss 6.852
Ep 1 (Step 000025): Train accuracy 0.118, Val accuracy 0.108
Ep 1 (Step 000030): Train loss 6.387, Val loss 6.673
Ep 1 (Step 000030): Train accuracy 0.138, Val accuracy 0.124
Ep 1 (Step 000035): Train loss 6.427, Val loss 6.540
Ep 1 (Step 000035): Train accuracy 0.132, Val accuracy 0.127
Ep 1 (Step 000040): Train loss 6.341, Val loss 6.462
Ep 1 (Step 000040): Train accurac

In [51]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [52]:
torch.manual_seed(123)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

idx = text_to_token_ids("The two men appeared out of nowhere", tokenizer)
idx = idx.to(device)

token_ids = generate(
    model=model,
    idx = idx,  # This is important
    max_new_tokens=150,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=30,
    temperature=1.4
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 The two men appeared out of nowhere,” said Harry could not 
in’t know as it. Hermione?” she’s 
on the room?” 

He and 
inny...” And we have you ... it the moment. Rowling 




“Hey have no 
Hermione was the door, “I don’t he did not just the 
the 
you’ve had 
him as 

But her voice. 
the Phoenix - J.K. She 
Mining of the 
s,”, though 
with 
sty.” 

“I mean as you have his office. All into the Order of his feet, there, 

She’s face, “Listen, though how — “They were a few the Phoenix - J.K. The truth for he’s 
n’ it and
